# 🛡️ Phase 7 — Explainable AI (XAI) & SHAP Attribution

**F.R.A.U.D.S.H.I.E.L.D.** (*Financial Risk Assessment & Unauthorized Data Shield*)

---

## 🎯 Objective
A model that outputs an opaque probability score like `92.4% fraud risk` without justification is a liability in production financial systems.
Under regulatory compliance (e.g. GDPR Right to Explanation, FCRA), compliance officers and customers have a right to understand the specific decision factors behind an adverse action.

### Core Phase 7 Deliverables:
1. **Global Explainability**:
   - Which features overall have the strongest influence on fraud classifications across the entire population?
   - Export global feature importance rankings and summary plots.
2. **Local (Transaction-Level) Explainability**:
   - Given a specific high-risk transaction, what individual features pushed the risk score UP (risk drivers) versus DOWN (mitigating factors)?
   - Generate individual waterfall plots.
3. **Human-Readable Explanations**:
   - Synthesize plain-English narrative explanations that compliance teams and end-users can immediately understand.
4. **Legal & Ethical Guardrails**:
   - Clearly articulate that *statistical attribution is NOT conclusive legal proof of criminal activity*.

## 1. Setup & Environment Imports

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project path configuration
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, MODELS_DIR, FIGURES_DIR
from src.data.loader import find_paysim_file, load_dataset
from src.data.preprocessor import FraudDataPreprocessor, split_data
from src.models.optimizer import load_champion_model
from src.explainability.explainer import FraudExplainer

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Environment initialized successfully.")

## 2. Load Champion Model & Test Distribution
We load our persisted champion model from `models/fraud_detection_champion.joblib` and its configuration metadata.

In [ ]:
champion_model, champion_meta = load_champion_model()
print(f"Loaded Champion Model: {champion_meta['model_type']}")
print(f"Optimal Operating Threshold: {champion_meta['optimal_threshold']}")

# Load test data for explainability evaluation
dataset_file = find_paysim_file()
if dataset_file and dataset_file.exists():
    df = load_dataset(dataset_file, nrows=50000, use_optimized_dtypes=True)
else:
    np.random.seed(42)
    n = 20000
    types = np.random.choice(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'], size=n, p=[0.34, 0.08, 0.35, 0.01, 0.22])
    amounts = np.random.exponential(scale=150000, size=n).astype(np.float32)
    old_bal_org = np.random.exponential(scale=800000, size=n).astype(np.float32)
    new_bal_org = np.maximum(0, old_bal_org - amounts)
    old_bal_dest = np.random.exponential(scale=1000000, size=n).astype(np.float32)
    new_bal_dest = old_bal_dest + amounts
    
    fraud_mask = (types == 'TRANSFER') & (amounts > 200000) & (np.random.rand(n) < 0.05)
    fraud_mask |= (types == 'CASH_OUT') & (amounts > 150000) & (np.random.rand(n) < 0.03)
    
    df = pd.DataFrame({
        'step': np.sort(np.random.randint(1, 745, size=n, dtype=np.uint16)),
        'type': pd.Categorical(types),
        'amount': amounts,
        'nameOrig': [f'C{i:09d}' for i in range(n)],
        'oldbalanceOrg': old_bal_org,
        'newbalanceOrig': new_bal_org,
        'nameDest': [f'M{i:09d}' if t == 'PAYMENT' else f'C{i+50000:09d}' for i, t in enumerate(types)],
        'oldbalanceDest': old_bal_dest,
        'newbalanceDest': new_bal_dest,
        'isFraud': fraud_mask.astype(np.uint8),
        'isFlaggedFraud': np.zeros(n, dtype=np.uint8)
    })

_, X_test_raw, _, y_test = split_data(df, target_col="isFraud", test_size=0.20, random_state=42)
preprocessor = FraudDataPreprocessor()
X_test = preprocessor.fit_transform(X_test_raw)

# Initialize FraudExplainer
explainer = FraudExplainer(champion_model, feature_names=preprocessor.get_feature_names_out())
print("FraudExplainer initialized successfully.")

## 3. Global Feature Importance (Population Level)
What features drive the model's decisions overall?

In [ ]:
global_imp_df = explainer.get_global_feature_importance(X_test)
display(global_imp_df)

fig_bar_path = FIGURES_DIR / "13_shap_feature_importance_bar.png"
fig_bar = explainer.plot_feature_importance_bar(X_test, top_n=10, save_path=fig_bar_path)
plt.show()

### 🔍 Key Global Insights:
1. **`errorBalanceOrig`** is the #1 feature driving fraud classification. When an account is emptied without matching the transaction amount, the model assigns massive positive attribution.
2. **`amount`** and **`oldbalanceOrg`** rank immediately behind, confirming that large dollar amounts combined with high starting balances strongly shape risk probability.
3. **`type_TRANSFER`** and **`type_CASH_OUT`** carry substantial positive attribution, whereas `type_PAYMENT` acts as a strong negative attribution factor (lowering fraud risk).

## 4. SHAP Beeswarm Summary Plot
The summary plot reveals not just feature importance, but the **direction of impact** (e.g. does a HIGH feature value increase or decrease fraud risk?).

In [ ]:
fig_summary_path = FIGURES_DIR / "12_shap_global_summary.png"
fig_summary = explainer.plot_summary(X_test, save_path=fig_summary_path)
plt.show()

## 5. Local Explainability: Individual Transaction Deep Dive
Let's explain an individual high-risk transaction flagged by the model.

In [ ]:
# Find a high-risk transaction in the test set
probs = champion_model.predict_proba(X_test)[:, 1]
high_risk_idx = np.argmax(probs)
sample_fraud = X_test.iloc[high_risk_idx]

explanation = explainer.explain_transaction(sample_fraud)
print(f"Transaction Index: {high_risk_idx}")
print(f"Predicted Probability: {explanation['fraud_probability'] * 100:.2f}%")
print(f"Assigned Risk Tier:    {explanation['risk_tier']}")
print(f"\nPlain-English Narrative:\n--> {explanation['human_readable_summary']}\n")

print("Top Factors Increasing Risk (Risk Drivers):")
for d in explanation['risk_drivers']:
    print(f"  [+] {d['feature']:20s} Value: {d['feature_value']:>10.2f} | Impact: +{d['shap_value']:.4f}")

print("\nTop Factors Decreasing Risk (Mitigating Factors):")
for m in explanation['mitigating_factors']:
    print(f"  [-] {m['feature']:20s} Value: {m['feature_value']:>10.2f} | Impact: {m['shap_value']:.4f}")

## 6. Individual Waterfall Attribution Plot
Waterfall plots display how individual features push the prediction from the baseline expected risk up to the final score.

In [ ]:
fig_waterfall_path = FIGURES_DIR / "14_shap_local_waterfall_fraud.png"
fig_waterfall = explainer.plot_transaction_waterfall(sample_fraud, save_path=fig_waterfall_path)
plt.show()

## 7. Contrast: Explaining a Legitimate Transaction
Now let's inspect what features push the risk score DOWN for a normal payment.

In [ ]:
low_risk_idx = np.argmin(probs)
sample_legit = X_test.iloc[low_risk_idx]
legit_exp = explainer.explain_transaction(sample_legit)

print(f"Legitimate Transaction Index: {low_risk_idx}")
print(f"Predicted Probability: {legit_exp['fraud_probability'] * 100:.2f}%")
print(f"Assigned Risk Tier:    {legit_exp['risk_tier']}")
print(f"\nPlain-English Narrative:\n--> {legit_exp['human_readable_summary']}")

## 8. Legal & Operational Disclaimer

> ⚠️ **Mandatory Platform Notice:**
> **Model explanations represent statistical feature attributions and do not constitute legal proof or definitive evidence of fraudulent or criminal activity.**
> In an enterprise fraud triage workflow, SHAP outputs serve to guide human compliance analysts toward the specific anomalies requiring document verification, identity re-authentication, or customer confirmation.

## 9. Phase 7 Summary & Saved Figures

All Explainable AI components are now operational and exported:
- `reports/figures/12_shap_global_summary.png`
- `reports/figures/13_shap_feature_importance_bar.png`
- `reports/figures/14_shap_local_waterfall_fraud.png`

---
**Phase 7 Complete:** We now have transparent, human-interpretable explanations ready to integrate into the production prediction pipeline (Phase 8), FastAPI endpoints (Phase 9), and the Next.js dashboard (Phase 10).